# LightRAG-only domain-filtering comparison (interactive)

Converted from `run_comparison_lightrag_only.py` (kept as-is alongside this notebook) so the domain-prediction and retrieval logic can be tweaked and re-run cell-by-cell instead of editing a `.py` file and re-running the whole script each time.

Two conditions, both through LightRAG (`run_aggregation`) only -- Hybrid RAG is skipped entirely for both, since it always searches the full index regardless of domain scope and would dilute a with/without-domain-filtering comparison:
- **baseline** -- `domains=None`, no filter at all
- **single_multi** -- domains from a single LLM call over the full domain list (no retrieval/RRF stage -- see "Domain prediction" below)

Structure: build each piece (domain prediction, then the full pipeline call) and sanity-check it on ONE question before running the full batch -- that's the fast way to see what's actually going wrong, rather than only ever looking at the final aggregate report.

In [41]:
from __future__ import annotations

import json
import logging
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

from pydantic import BaseModel

CHATBOT_ROOT = Path.cwd().resolve()
for _candidate in [CHATBOT_ROOT, *CHATBOT_ROOT.parents]:
    if (_candidate / "src" / "pipeline").exists():
        CHATBOT_ROOT = _candidate
        break
if str(CHATBOT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHATBOT_ROOT))
print("CHATBOT_ROOT =", CHATBOT_ROOT)

from src.pipeline.query_understanding import analyze_query
from src.pipeline.lightrag import run_aggregation
from src.pipeline.fusion import reciprocal_rank_fusion, rerank_chunks
from src.pipeline.answer import generate_answer
from src.pipeline.clients import call_json, validate_model

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("agent_flow.domain_eval.lightrag_only_nb")

CHATBOT_ROOT = /Users/senghok/Projects/gds_tender_test/chatbot


In [42]:
DOMAIN_DATA_DIR = CHATBOT_ROOT / "domain_data"
QUESTIONS_PATH = DOMAIN_DATA_DIR / "domain_eval_questions.json"
RESULTS_DIR = CHATBOT_ROOT / "domain_data" / "eval_results"

RRF_POOL = 50
FINAL_CONTEXT_CHUNKS = 20
MAX_MULTI_DOMAINS = 2

In [43]:
def load_questions(limit: Optional[int] = None) -> List[Dict[str, str]]:
    data = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
    qs = data["questions"]
    return qs[:limit] if limit else qs


def save_results(results: List[Dict[str, Any]], path: Path) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info("Wrote %d results -> %s", len(results), path)


all_questions = load_questions()
print(f"{len(all_questions)} eval questions loaded")
all_questions[:3]

78 eval questions loaded


[{'id': 'PROCUREMENT-01',
  'domain': 'PROCUREMENT',
  'question': 'What are the requirements for vendor selection and material sourcing under this tender?'},
 {'id': 'PROCUREMENT-02',
  'domain': 'PROCUREMENT',
  'question': 'What RFQ and bidding process must the Contractor follow when procuring long-lead equipment?'},
 {'id': 'PROCESS_HSED-01',
  'domain': 'PROCESS & HSED (SAFETY IN DESIGN)',
  'question': 'What HAZOP and HAZID reviews are required during the process design phase?'}]

## Domain prediction (LLM-only: single call over the full domain list)

No retrieval stage at all -- the LLM sees every domain's name + description in one prompt and picks directly. Simpler than RRF+verifier (no embeddings, no keyword matching, no shortlist), at the cost of asking the LLM to discriminate among ~30 options at once instead of a pre-narrowed few. Edit `_VERIFIER_USER` / `predict_single_multi` below and re-run the sanity-check cell to iterate.

In [70]:
@lru_cache(maxsize=1)
def _domain_descriptions() -> Dict[str, str]:
    return json.loads((DOMAIN_DATA_DIR / "domain_description_ROC_INPEX.json").read_text(encoding="utf-8"))


@lru_cache(maxsize=1)
def _domain_keywords() -> Dict[str, List[str]]:
    return json.loads((DOMAIN_DATA_DIR / "domain_keywords_ROC_INPEX.json").read_text(encoding="utf-8"))


print(f"{len(_domain_descriptions())} domains loaded from domain_description_ROC_INPEX.json")

40 domains loaded from domain_description_ROC_INPEX.json


In [100]:
class DomainScanEntry(BaseModel):
    domain: str
    fits: bool
    note: str = ""


class SingleMultiVerification(BaseModel):
    domain_scan: List[DomainScanEntry] = []
    scope: Literal["single", "multi"]
    domains: List[str]
    reasoning: str = ""


_VERIFIER_SYS = (
    "You are a senior engineering document classifier for EPC (Engineering, Procurement, "
    "Construction) tender and ITB (Invitation to Bid) documents. You determine which "
    "discipline domain(s) a question falls under, drawing on deep familiarity with process "
    "and detailed engineering, procurement, construction, project management, and the "
    "commercial/legal practices of tendering. Respond with strict JSON only -- no prose, no "
    "markdown, and no explanation outside the JSON object."
)

_VERIFIER_USER = """Classify the question below into the domain(s) it belongs to, using ONLY the
domain list provided below -- this is the complete list of available domains, not a shortlist.

Process to follow (do this before deciding):
1. Go through the domain list below ONE BY ONE, in order. For each domain, read its description
   and keywords and check whether the question's CORE SUBJECT -- the specific equipment, system,
   or discipline it is actually about -- matches that domain. Record a brief true/false + one-line
   note for every domain in "domain_scan".
2. Do not classify based on a generic action/aspect word alone (e.g. "design requirements",
   "procedures", "specifications", "testing", "management" appear across nearly every domain and
   carry no discriminating signal by themselves) -- match the SUBJECT, not the verb.
3. Do not classify a domain just because a word in the question happens to match or resemble that
   domain's NAME, or happens to be an artifact/keyword loosely ASSOCIATED with that domain -- check
   whether the question genuinely requires that domain's subject-matter knowledge to answer, not a
   surface word coincidence. A question about "quality sample analysis in a laboratory" is about
   LABORATORY (physical testing facilities), not QUALITY MANAGEMENT (QA/QC processes, audits),
   even though the word "quality" appears. A question about how many exhibits a document package
   contains is not about LEGAL / CONTRACT just because exhibits are contract artifacts -- it is a
   meta-question about the package's own structure, not a legal/contractual matter.
4. Only after completing the scan, decide:
   - scope = "single": exactly one domain clearly fits the core subject. If, after scanning every
     domain, NONE of the specific domains genuinely fit -- the question's core subject is not a
     substantive engineering, procurement, or commercial discipline at all -- select "GENERAL" as
     the domain instead of force-fitting the closest weak match. This includes, but is not limited
     to: broad overviews of the whole tender; purely administrative/meta questions about the
     document package itself (e.g. how many exhibits it has, what language or units it uses, how
     it is organized); and any other question where a specific domain would only be a tangential,
     word-level stretch rather than a genuine subject-matter match. Never pick a specific domain
     just because it is the "least wrong" option -- if none of them genuinely fit, GENERAL is the
     correct and expected answer, not a fallback of last resort.
   - scope = "multi": answering the question fully genuinely requires combining information from
     more than one SPECIFIC domain (up to {max_multi}) -- not because a term loosely touches
     another domain in passing. Prefer "single" when in doubt; use "multi" only when it is clearly
     warranted. GENERAL is never combined with other domains -- if GENERAL applies, it is always
     the sole entry in "domains" with scope = "single".
- This question is already confirmed to be in scope for one of these domains (including GENERAL)
  -- you must always select at least one domain; never return an empty list or decline to classify.
- Each entry in "domains" must be copied EXACTLY as it appears below (same spelling, spacing, and
  punctuation) -- any name that does not match exactly will be discarded by the calling code.

Examples of correct final reasoning (illustrative domain names only -- always classify against the
actual domain list given to you below, not the names used in these examples; your actual answer
must also include a full "domain_scan" covering every domain listed below, omitted here for
brevity):

Question: "What are the design requirements for the fired heaters in the process package?"
Reasoning: the core subject is "fired heaters" -- a specific, named equipment type -- not the
generic phrase "design requirements" or the word "process".
Correct final fields: {{"scope": "single", "domains": ["FIRED EQUIPMENT"], "reasoning": "The question is specifically about fired heaters, a named equipment type, not a general process-design topic."}}

Question: "What testing and inspection procedures apply to pressure vessel fabrication?"
Reasoning: the core subject is "pressure vessel fabrication" -- not "testing and inspection
procedures" alone, which is a generic aspect that could apply to almost any equipment.
Correct final fields: {{"scope": "single", "domains": ["PRESSURE VESSELS"], "reasoning": "Fabrication and testing here concern pressure vessels specifically, not quality management in general."}}

Question: "What vendor qualification criteria and delivery schedule commitments apply to rotating equipment packages?"
Reasoning: this genuinely spans two domains -- vendor qualification and delivery are procurement
concerns, and rotating equipment is its own engineering discipline -- neither answers the question
alone.
Correct final fields: {{"scope": "multi", "domains": ["PROCUREMENT", "ROTATING EQUIPMENT"], "reasoning": "The question asks about both vendor/procurement terms and rotating-equipment-specific requirements."}}

Question: "What laboratory testing facilities are required for quality sample analysis?"
Reasoning: the core subject is "laboratory testing facilities" -- the word "quality" here describes
the PURPOSE of the sample analysis, it does not signal the QUALITY MANAGEMENT domain (which covers
QA/QC processes, audits, and certification programs -- not physical lab facilities or equipment).
Correct final fields: {{"scope": "single", "domains": ["LABORATORY"], "reasoning": "The question asks about physical laboratory testing facilities, not quality-assurance processes or audits."}}

Question: "What is the overall scope and objectives of this tender package?"
Reasoning: this question is broad and cross-disciplinary -- it asks about the tender as a whole,
not about any single engineering, procurement, or commercial discipline.
Correct final fields: {{"scope": "single", "domains": ["GENERAL"], "reasoning": "The question asks for a broad overview of the tender, not a topic specific to any single domain."}}

Question: "How many exhibits and schedules are included in this ITB?"
Reasoning: this is NOT a broad overview question, but it still has no genuine discipline at its
core -- it is a meta-question about the document package's own structure/count. "Exhibits and
schedules" being contract artifacts does not make this a LEGAL / CONTRACT question -- answering it
requires no legal or contractual subject-matter knowledge at all, just counting.
Correct final fields: {{"scope": "single", "domains": ["GENERAL"], "reasoning": "The question asks about the document package's own structure/count, not a legal, contractual, or any other specific discipline."}}

Available domains (name: description (keywords)):
{domain_block}

Question: {question}

Return STRICT JSON in exactly this shape, with no additional keys. "domain_scan" must contain
exactly one entry per domain listed above, in the same order:
{{"domain_scan": [{{"domain": "<exact domain name>", "fits": true|false, "note": "<very short reason, a few words>"}}, ...], "scope": "single|multi", "domains": ["<exact domain name from the list above>", ...], "reasoning": "<one concise sentence citing the specific part of the question that justifies this choice>"}}
"""


def _domain_block() -> str:
    descriptions = _domain_descriptions()
    keywords = _domain_keywords()
    lines = []
    for name, desc in descriptions.items():
        kw = keywords.get(name) or []
        kw_str = f" (keywords: {', '.join(kw)})" if kw else ""
        lines.append(f"- {name}: {desc}{kw_str}")
    return "\n".join(lines)


def predict_single_multi(question: str) -> Dict[str, Any]:
    """LLM-only domain prediction: no retrieval/fusion stage -- the LLM scans every
    domain's name + description + keywords one by one (via the "domain_scan" field,
    generated before the final answer within the same JSON call) and only then decides.
    "GENERAL" is one of the scanned domains (already present in domain_description_ROC_INPEX.json)
    -- it is the correct answer when the question doesn't genuinely concern any specific domain.

    Note: this makes each call noticeably longer (one domain_scan entry per domain,
    ~40 domains) than a direct-answer prompt would be -- more latency/cost per
    question in exchange for forcing the systematic per-domain comparison.
    """
    all_domains = list(_domain_descriptions().keys())
    domain_block = _domain_block()

    domain_scan: List[Dict[str, Any]] = []
    try:
        data = call_json(_VERIFIER_SYS, _VERIFIER_USER.format(
            domain_block=domain_block, question=question, max_multi=MAX_MULTI_DOMAINS))
        pred = validate_model(data, SingleMultiVerification)
        scope, domains, reasoning = pred.scope, list(pred.domains), pred.reasoning
        domain_scan = [e.model_dump() if hasattr(e, "model_dump") else e.dict() for e in pred.domain_scan]
    except Exception as exc:
        # No retrieval ranking to fall back on here -- there's no signal for which
        # domain is most plausible if the LLM call itself fails, so this just picks
        # the first domain in the description file, which is an arbitrary guess,
        # not a ranked one. Worth knowing if fallback rate is high.
        logger.warning("predict_single_multi (LLM-only) fallback (%s) -> arbitrary first domain", exc)
        scope, domains, reasoning = "single", all_domains[:1], f"fallback: {exc}"

    domains = [d for d in domains if d in all_domains]
    if not domains:
        scope, domains = "single", all_domains[:1]
    elif "GENERAL" in domains and len(domains) > 1:
        # GENERAL means "nothing specific fits" -- never valid combined with a
        # specific domain, regardless of what scope the model actually returned.
        scope, domains = "single", ["GENERAL"]
    elif scope == "single" and len(domains) > 1:
        domains = domains[:1]
    elif scope == "multi":
        domains = domains[:MAX_MULTI_DOMAINS]

    return {"scope": scope, "domains": domains, "reasoning": reasoning,
            "top_candidates": all_domains, "domain_scan": domain_scan}

In [94]:
domains

['LEGAL / CONTRACT', 'FINANCE / TAX']

### Sanity check: domain prediction alone, on ONE question

Fast (one LLM call, no embeddings/keyword matching) -- use this to iterate on the verifier prompt before touching anything downstream.

In [101]:
{
  "questions": [
    {"id": "PROCUREMENT-01", "domain": "PROCUREMENT", "question": "What are the requirements for vendor selection and material sourcing under this tender?"},
    {"id": "PROCUREMENT-02", "domain": "PROCUREMENT", "question": "What RFQ and bidding process must the Contractor follow when procuring long-lead equipment?"},

    {"id": "PROCESS_HSED-01", "domain": "PROCESS & HSED (SAFETY IN DESIGN)", "question": "What HAZOP and HAZID reviews are required during the process design phase?"},
    {"id": "PROCESS_HSED-02", "domain": "PROCESS & HSED (SAFETY IN DESIGN)", "question": "What process design philosophy and safety-in-design safeguards apply to the AGRU upgrade?"},

    {"id": "LEGAL_CONTRACT-01", "domain": "LEGAL / CONTRACT", "question": "What are the Contractor's obligations regarding dispute resolution?"},
    {"id": "LEGAL_CONTRACT-02", "domain": "LEGAL / CONTRACT", "question": "What indemnity and liability provisions apply to the Contractor under this contract?"},

    {"id": "FINANCE_TAX-01", "domain": "FINANCE / TAX", "question": "What are the payment terms and invoicing requirements for the Contractor?"},
    {"id": "FINANCE_TAX-02", "domain": "FINANCE / TAX", "question": "What tax compliance and fiscal reporting obligations does the Contractor have?"},

    {"id": "INSURANCE-01", "domain": "INSURANCE", "question": "What insurance coverage must the Contractor obtain and maintain during the project?"},
    {"id": "INSURANCE-02", "domain": "INSURANCE", "question": "What is the minimum third-party liability insurance limit required per occurrence?"},

    {"id": "IP_NDA-01", "domain": "INTELLECTUAL PROPERTY / NDA", "question": "What confidentiality obligations apply to information shared under this tender?"},
    {"id": "IP_NDA-02", "domain": "INTELLECTUAL PROPERTY / NDA", "question": "Who owns the intellectual property rights for designs developed during the contract?"},

    {"id": "INFO_MGMT-01", "domain": "INFORMATION MANAGEMENT", "question": "What document control and EDMS requirements apply to project deliverables?"},
    {"id": "INFO_MGMT-02", "domain": "INFORMATION MANAGEMENT", "question": "What are the data and information handover requirements at contract close-out?"},

    {"id": "CYBERSECURITY-01", "domain": "CYBERSECURITY", "question": "What cybersecurity requirements apply to the process control systems network?"},
    {"id": "CYBERSECURITY-02", "domain": "CYBERSECURITY", "question": "What access control and authentication measures are required for project IT systems?"},

    {"id": "ROTATING_EQUIPMENT-01", "domain": "ROTATING EQUIPMENT", "question": "What are the specification requirements for the CO2 compressor?"},
    {"id": "ROTATING_EQUIPMENT-02", "domain": "ROTATING EQUIPMENT", "question": "What maintenance and supplier qualification strategy is required for pumps and turbines?"},

    {"id": "PRESSURE_VESSELS-01", "domain": "PRESSURE VESSELS", "question": "What ASME code compliance is required for the pressure vessels in the AGRU upgrade?"},
    {"id": "PRESSURE_VESSELS-02", "domain": "PRESSURE VESSELS", "question": "What inspection requirements apply to the heat exchangers and columns?"},

    {"id": "PIPING_INSTALLATION-01", "domain": "PIPING INSTALLATION", "question": "What welding standards apply to piping installation on this project?"},
    {"id": "PIPING_INSTALLATION-02", "domain": "PIPING INSTALLATION", "question": "What are the 'Issued for Construction' deliverable requirements for piping isometrics?"},

    {"id": "PIPELINES-01", "domain": "PIPELINES", "question": "What integrity management requirements apply to the pipeline systems?"},
    {"id": "PIPELINES-02", "domain": "PIPELINES", "question": "What pigging operation requirements apply to the transmission pipelines?"},

    {"id": "PACKAGE-01", "domain": "PACKAGE", "question": "What are the requirements for vendor-supplied packaged equipment skids?"},
    {"id": "PACKAGE-02", "domain": "PACKAGE", "question": "What documentation must be provided for pre-assembled modular equipment packages?"},

    {"id": "FIRED_EQUIPMENT-01", "domain": "FIRED EQUIPMENT", "question": "What are the design requirements for the fired heaters in the process package?"},
    {"id": "FIRED_EQUIPMENT-02", "domain": "FIRED EQUIPMENT", "question": "What combustion and burner design standards apply to the boilers?"},

    {"id": "ONSHORE_BUILDING-01", "domain": "ONSHORE BUILDING/ ARCHITECTURE", "question": "What civil foundation requirements apply to the onshore buildings in this project?"},
    {"id": "ONSHORE_BUILDING-02", "domain": "ONSHORE BUILDING/ ARCHITECTURE", "question": "What architectural design standards apply to the onshore facility layout?"},

    {"id": "CONTROL_SYSTEMS-01", "domain": "CONTROL SYSTEMS", "question": "What DCS and SCADA requirements apply to the process control system?"},
    {"id": "CONTROL_SYSTEMS-02", "domain": "CONTROL SYSTEMS", "question": "What HMI design standards must the Contractor follow for the automation system?"},

    {"id": "TELECOM-01", "domain": "TELECOMMUNICATIONS", "question": "What telecommunications infrastructure is required for the site?"},
    {"id": "TELECOM-02", "domain": "TELECOMMUNICATIONS", "question": "What radio and fiber optic communication systems must be installed?"},

    {"id": "LABORATORY-01", "domain": "LABORATORY", "question": "What laboratory testing facilities are required for quality sample analysis?"},
    {"id": "LABORATORY-02", "domain": "LABORATORY", "question": "What equipment must be provided for the on-site laboratory?"},

    {"id": "SUBCONTRACTING-01", "domain": "SUBCONTRACTING", "question": "What are the requirements for managing subcontractors on this project?"},
    {"id": "SUBCONTRACTING-02", "domain": "SUBCONTRACTING", "question": "What yard management obligations apply to subcontracted fabrication work?"},

    {"id": "OFFSHORE_MOBILE-01", "domain": "OFFSHORE MOBILE SYSTEM", "question": "What design documentation is required for mobile offshore units used in this project?"},
    {"id": "OFFSHORE_MOBILE-02", "domain": "OFFSHORE MOBILE SYSTEM", "question": "What requirements apply to floating production systems used on this project?"},

    {"id": "OFFSHORE_BUILDING-01", "domain": "OFFSHORE BUILDING/ ARCHITECTURE", "question": "What are the design requirements for offshore living quarters and accommodation modules?"},
    {"id": "OFFSHORE_BUILDING-02", "domain": "OFFSHORE BUILDING/ ARCHITECTURE", "question": "What helideck layout standards apply to the offshore facility?"},

    {"id": "CIVIL_STRUCTURAL-01", "domain": "CIVIL / STRUCTURAL", "question": "What structural engineering standards apply to the concrete and steel structures?"},
    {"id": "CIVIL_STRUCTURAL-02", "domain": "CIVIL / STRUCTURAL", "question": "What foundation design requirements apply to the new structures?"},

    {"id": "ELECTRICAL-01", "domain": "ELECTRICAL", "question": "What electrical power distribution requirements apply to the new equipment?"},
    {"id": "ELECTRICAL-02", "domain": "ELECTRICAL", "question": "What switchgear and transformer specifications are required for the substation?"},

    {"id": "INSTRUMENTATION-01", "domain": "INSTRUMENTATION", "question": "What field device and transmitter specifications apply to the instrumentation scope?"},
    {"id": "INSTRUMENTATION-02", "domain": "INSTRUMENTATION", "question": "What calibration procedures are required for the measurement instruments?"},

    {"id": "HVAC-01", "domain": "HVAC", "question": "What HVAC system design requirements apply to the control room?"},
    {"id": "HVAC-02", "domain": "HVAC", "question": "What ductwork and chiller specifications are required for the facility?"},

    {"id": "NAVAL_ARCHITECTURE-01", "domain": "NAVAL ARCHITECTURE", "question": "What stability analysis is required for the floating structure?"},
    {"id": "NAVAL_ARCHITECTURE-02", "domain": "NAVAL ARCHITECTURE", "question": "What hydrodynamic studies must be performed for the vessel design?"},

    {"id": "OFFSHORE_STRUCTURE-01", "domain": "OFFSHORE STRUCTURE", "question": "What foundation analysis is required for the offshore platform jacket?"},
    {"id": "OFFSHORE_STRUCTURE-02", "domain": "OFFSHORE STRUCTURE", "question": "What mooring system requirements apply to the offshore structure?"},

    {"id": "MATERIALS_TECHNOLOGY-01", "domain": "MATERIALS AND TECHNOLOGY", "question": "What material selection requirements apply to corrosion-resistant piping?"},
    {"id": "MATERIALS_TECHNOLOGY-02", "domain": "MATERIALS AND TECHNOLOGY", "question": "What coating specifications are required for the equipment?"},

    {"id": "ADVANCED_SYSTEMS-01", "domain": "ADVANCED SYSTEMS", "question": "What digital optimization or AI/IoT technologies are required for this project?"},
    {"id": "ADVANCED_SYSTEMS-02", "domain": "ADVANCED SYSTEMS", "question": "What smart system integration is expected under this contract?"},

    {"id": "INFO_TECH-01", "domain": "INFORMATION TECHNOLOGY", "question": "What IT infrastructure and server requirements apply to the project systems?"},
    {"id": "INFO_TECH-02", "domain": "INFORMATION TECHNOLOGY", "question": "What software platforms must the Contractor use for project administration?"},

    {"id": "PROJECT_EXEC_SYSTEMS-01", "domain": "PROJECT EXECUTION SYSTEMS", "question": "What 3D modeling system (Aveva) requirements apply to the engineering deliverables?"},
    {"id": "PROJECT_EXEC_SYSTEMS-02", "domain": "PROJECT EXECUTION SYSTEMS", "question": "What EDMS and collaborative tools must be set up for the project?"},

    {"id": "COSTS_CONTROL-01", "domain": "COSTS CONTROL", "question": "What cost tracking and variance analysis requirements apply to the Contractor?"},
    {"id": "COSTS_CONTROL-02", "domain": "COSTS CONTROL", "question": "What are the budget forecasting obligations under this contract?"},

    {"id": "RISK_MANAGEMENT-01", "domain": "RISK MANAGEMENT", "question": "What risk register management process must the Contractor follow?"},
    {"id": "RISK_MANAGEMENT-02", "domain": "RISK MANAGEMENT", "question": "What contingency planning is required for identified project risks?"},

    {"id": "DOCUMENT_CONTROL-01", "domain": "DOCUMENT CONTROL", "question": "What document revision and version control requirements apply to deliverables?"},
    {"id": "DOCUMENT_CONTROL-02", "domain": "DOCUMENT CONTROL", "question": "What are the transmittal and document register requirements for this project?"},

    {"id": "HSE_MANAGEMENT-01", "domain": "HSE MANAGEMENT", "question": "What emergency response procedures must the Contractor implement?"},
    {"id": "HSE_MANAGEMENT-02", "domain": "HSE MANAGEMENT", "question": "What incident management and reporting requirements apply to the Contractor?"},

    {"id": "SCHEDULE_MANAGEMENT-01", "domain": "SCHEDULE MANAGEMENT", "question": "What are the critical path and baseline schedule requirements for this project?"},
    {"id": "SCHEDULE_MANAGEMENT-02", "domain": "SCHEDULE MANAGEMENT", "question": "What progress tracking and Gantt chart requirements apply to the Contractor?"},

    {"id": "QUALITY_MANAGEMENT-01", "domain": "QUALITY MANAGEMENT", "question": "What QA/QC procedures must the Contractor follow?"},
    {"id": "QUALITY_MANAGEMENT-02", "domain": "QUALITY MANAGEMENT", "question": "What non-conformance report (NCR) management process is required?"},

    {"id": "WEIGHT_CONTROL-01", "domain": "WEIGHT CONTROL AND MANAGEMENT", "question": "What weight estimation and center-of-gravity reporting requirements apply to the modules?"},
    {"id": "WEIGHT_CONTROL-02", "domain": "WEIGHT CONTROL AND MANAGEMENT", "question": "What weight reduction strategy must the Contractor follow for the floating asset?"},

    {"id": "TRANSPORT_INSTALLATION-01", "domain": "TRANSPORT AND INSTALLATION", "question": "What heavy-lift and marine transport requirements apply to module installation?"},
    {"id": "TRANSPORT_INSTALLATION-02", "domain": "TRANSPORT AND INSTALLATION", "question": "What loadout and seafastening procedures must the Contractor follow?"}
  ]
}


{'questions': [{'id': 'PROCUREMENT-01',
   'domain': 'PROCUREMENT',
   'question': 'What are the requirements for vendor selection and material sourcing under this tender?'},
  {'id': 'PROCUREMENT-02',
   'domain': 'PROCUREMENT',
   'question': 'What RFQ and bidding process must the Contractor follow when procuring long-lead equipment?'},
  {'id': 'PROCESS_HSED-01',
   'domain': 'PROCESS & HSED (SAFETY IN DESIGN)',
   'question': 'What HAZOP and HAZID reviews are required during the process design phase?'},
  {'id': 'PROCESS_HSED-02',
   'domain': 'PROCESS & HSED (SAFETY IN DESIGN)',
   'question': 'What process design philosophy and safety-in-design safeguards apply to the AGRU upgrade?'},
  {'id': 'LEGAL_CONTRACT-01',
   'domain': 'LEGAL / CONTRACT',
   'question': "What are the Contractor's obligations regarding dispute resolution?"},
  {'id': 'LEGAL_CONTRACT-02',
   'domain': 'LEGAL / CONTRACT',
   'question': 'What indemnity and liability provisions apply to the Contractor under t

In [103]:
sample_item = "What currency should bid amounts be submitted in?"
print("question:", sample_item)
# print("expected_domain:", sample_item["domain"])
print()
pred = predict_single_multi(sample_item)
pred

question: What currency should bid amounts be submitted in?



2026-07-21 11:30:25,128 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


{'scope': 'multi',
 'domains': ['PROCUREMENT', 'LEGAL / CONTRACT'],
 'reasoning': 'The question concerns currency for bid amounts, which is relevant to procurement terms and contract specifications.',
 'top_candidates': ['PROCUREMENT',
  'PROCESS & HSED (SAFETY IN DESIGN)',
  'LEGAL / CONTRACT',
  'FINANCE / TAX',
  'INSURANCE',
  'INTELLECTUAL PROPERTY / NDA',
  'INFORMATION MANAGEMENT',
  'CYBERSECURITY',
  'ROTATING EQUIPMENT',
  'PRESSURE VESSELS',
  'PIPING INSTALLATION',
  'PIPELINES',
  'PACKAGE',
  'FIRED EQUIPMENT',
  'ONSHORE BUILDING/ ARCHITECTURE',
  'CONTROL SYSTEMS',
  'TELECOMMUNICATIONS',
  'LABORATORY',
  'SUBCONTRACTING',
  'OFFSHORE MOBILE SYSTEM',
  'OFFSHORE BUILDING/ ARCHITECTURE',
  'CIVIL / STRUCTURAL',
  'ELECTRICAL',
  'INSTRUMENTATION',
  'HVAC',
  'NAVAL ARCHITECTURE',
  'OFFSHORE STRUCTURE',
  'MATERIALS AND TECHNOLOGY',
  'ADVANCED SYSTEMS',
  'INFORMATION TECHNOLOGY',
  'PROJECT EXECUTION SYSTEMS',
  'COSTS CONTROL',
  'RISK MANAGEMENT',
  'DOCUMENT CONTR

### Domain prediction accuracy over ALL questions

Runs `predict_single_multi` on every question in `all_questions` (parallelized -- these are plain LLM calls, no reranker involved, so unlike the full pipeline this is safe to run concurrently) and separates results into `correct_predictions` / `wrong_predictions`, comparing against each question's labeled `domain`.

In [78]:
def _score_question(item: Dict[str, str]) -> Dict[str, Any]:
    expected = item["domain"]
    pred = predict_single_multi(item["question"])
    predicted = pred.get("domains") or []
    return {
        "id": item["id"],
        "question": item["question"],
        "expected": expected,
        "predicted": predicted,
        "scope": pred.get("scope"),
        "reasoning": pred.get("reasoning"),
        "domain_scan": pred.get("domain_scan"),
        "correct": expected in predicted,
    }


DOMAIN_TEST_MAX_WORKERS = 4  # pure LLM calls, no reranker involved -- safe to parallelize

all_domain_results: List[Optional[Dict[str, Any]]] = [None] * len(all_questions)
with ThreadPoolExecutor(max_workers=DOMAIN_TEST_MAX_WORKERS) as ex:
    futures = {ex.submit(_score_question, item): i for i, item in enumerate(all_questions)}
    done = 0
    for future in as_completed(futures):
        i = futures[future]
        all_domain_results[i] = future.result()
        done += 1
        r = all_domain_results[i]
        mark = "OK" if r["correct"] else "WRONG"
        print(f"[{done}/{len(all_questions)}] {mark:>5}  {r['id']}: expected={r['expected']!r} predicted={r['predicted']}")

correct_predictions = [r for r in all_domain_results if r["correct"]]
wrong_predictions = [r for r in all_domain_results if not r["correct"]]

print()
print("=" * 72)
print(f"Accuracy: {len(correct_predictions)}/{len(all_domain_results)} "
      f"({round(100 * len(correct_predictions) / len(all_domain_results), 1)}%)")
print("=" * 72)
print()
print(f"Wrong ({len(wrong_predictions)}):")
for r in wrong_predictions:
    print(f"- {r['id']}: expected={r['expected']!r} predicted={r['predicted']} "
          f"scope={r['scope']} reasoning={r['reasoning']!r}")

2026-07-21 10:52:04,572 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


[1/78]    OK  PROCESS_HSED-01: expected='PROCESS & HSED (SAFETY IN DESIGN)' predicted=['PROCESS & HSED (SAFETY IN DESIGN)']


2026-07-21 10:52:04,980 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-07-21 10:52:05,034 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-07-21 10:52:05,111 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


[2/78]    OK  PROCUREMENT-02: expected='PROCUREMENT' predicted=['PROCUREMENT']
[3/78]    OK  PROCUREMENT-01: expected='PROCUREMENT' predicted=['PROCUREMENT']
[4/78]    OK  PROCESS_HSED-02: expected='PROCESS & HSED (SAFETY IN DESIGN)' predicted=['PROCESS & HSED (SAFETY IN DESIGN)']


2026-07-21 10:52:11,842 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


...

[76/78]    OK  WEIGHT_CONTROL-02: expected='WEIGHT CONTROL AND MANAGEMENT' predicted=['WEIGHT CONTROL AND MANAGEMENT']


2026-07-21 10:54:26,606 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


[77/78]    OK  TRANSPORT_INSTALLATION-01: expected='TRANSPORT AND INSTALLATION' predicted=['TRANSPORT AND INSTALLATION']


2026-07-21 10:54:28,181 INFO HTTP Request: POST https://apis-dev.lab.technipenergies.com/sales/llmfortendering/cognitiveservices/openai/deployments/gpt-4o-llm4t/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


[78/78]    OK  TRANSPORT_INSTALLATION-02: expected='TRANSPORT AND INSTALLATION' predicted=['TRANSPORT AND INSTALLATION']

Accuracy: 76/78 (97.4%)

Wrong (2):
- INFO_TECH-02: expected='INFORMATION TECHNOLOGY' predicted=['INFORMATION MANAGEMENT', 'PROJECT EXECUTION SYSTEMS'] scope=multi reasoning='The question involves both data handling systems and specific project administration platforms.'
- COSTS_CONTROL-02: expected='COSTS CONTROL' predicted=['LEGAL / CONTRACT', 'FINANCE / TAX'] scope=multi reasoning='The question concerns budget forecasting (finance) obligations under a contract (legal/contract).'
